<a href="https://colab.research.google.com/github/iship-itb14/ishitp-itb14.github.io/blob/main/Penguin_Data_Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
MAPPPD Penguin Population Data — Cleaning, Preprocessing & EDA
Species: Gentoo (gentoo penguin), King (king penguin), Macaroni (macaroni penguin)
Source:  MAPPPD direct download (penguinmap.com) — DownloadAll.csv
"""

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import kruskal
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
import os
import requests

OUT = "Penguin_plots"
os.makedirs(OUT, exist_ok=True)

sns.set_theme(style="whitegrid", font_scale=1.15)
PALETTE = {"gentoo penguin": "#1b7837", "macaroni penguin": "#762a83", "king penguin": "#e08214"}
LABELS  = {"gentoo penguin": "Gentoo",  "macaroni penguin": "Macaroni", "king penguin": "King"}
SPECIES = list(PALETTE.keys())


# 1. DATA COLLECTION

print("── Step 1: Data Collection ──")

import requests, os, io

# ── DATA COLLECTION via live download
MAPPPD_URL   = "https://www.penguinmap.com/mapppd/DownloadAll/"
CACHE_PATH   = "DownloadAll.csv"

def load_mapppd(url=MAPPPD_URL, cache=CACHE_PATH, force_refresh=False):
    """
    Download the MAPPPD 'DownloadAll' CSV from penguinmap.com.
    Re-uses a local cache unless force_refresh=True.
    """
    if os.path.exists(cache) and not force_refresh:
        print(f"  Using cached file: {cache}")
        return pd.read_csv(cache)

    print(f"  Fetching live data from {url} …")
    headers = {"User-Agent": "Mozilla/5.0"}   # required; bare requests get blocked
    resp = requests.get(url, headers=headers, timeout=60)
    resp.raise_for_status()                   # raises if 4xx/5xx

    raw = pd.read_csv(io.StringIO(resp.text))
    raw.to_csv(cache, index=False)            # save cache
    print(f"  Downloaded {len(raw):,} rows → cached to {cache}")
    return raw

raw = load_mapppd()
print(f"  Raw shape  : {raw.shape}")
print(f"  Columns    : {raw.columns.tolist()}")
print(f"  All species: {raw['common_name'].unique().tolist()}")


# 2. BEFORE SNAPSHOT

before_snapshot = raw[raw["common_name"].isin(SPECIES)].copy()
print(f"\n── BEFORE snapshot (3 species) ──")
print(f"  Rows          : {len(before_snapshot):,}")
print(f"  Missing values:\n{before_snapshot.isnull().sum().to_string()}")
print(before_snapshot[["common_name","year","site_name","penguin_count","count_type","vantage"]].head(5).to_string())


# 3. FILTER TO 3 SPECIES + RENAME
print("\n── Step 2: Cleaning ──")

df = raw[raw["common_name"].isin(SPECIES)].copy()
df.rename(columns={
    "longitude_epsg_4326": "lon",
    "latitude_epsg_4326":  "lat",
}, inplace=True)
print(f"  After species filter (Gentoo / King / Macaroni): {len(df):,} rows")


# 4. CLEANING

# 4a. Year filter — MAPPPD systematic coverage starts ~1979
n0 = len(df)
df = df[df["year"].between(1979, 2025)]
print(f"  Year filter (1979–2025)   : dropped {n0 - len(df)} rows")

# 4b. Drop rows with missing penguin_count
n1 = len(df)
df.dropna(subset=["penguin_count"], inplace=True)
print(f"  Dropped missing counts    : {n1 - len(df)} rows")

# 4c. Remove zero counts (uninformative for population analysis)
n2 = len(df)
df = df[df["penguin_count"] > 0]
print(f"  Removed zero counts       : {n2 - len(df)} rows")

# 4d. Duplicates — same site + species + year + count_type
n3 = len(df)
df.drop_duplicates(subset=["site_id", "common_name", "year", "count_type"], inplace=True)
print(f"  Removed duplicates        : {n3 - len(df)} rows")

# 4e. Outlier detection per species (3×IQR)
outlier_mask = pd.Series(False, index=df.index)
for sp in SPECIES:
    mask = df["common_name"] == sp
    q1 = df.loc[mask, "penguin_count"].quantile(0.25)
    q3 = df.loc[mask, "penguin_count"].quantile(0.75)
    iqr = q3 - q1
    flag = mask & ((df["penguin_count"] < q1 - 3*iqr) | (df["penguin_count"] > q3 + 3*iqr))
    print(f"  Outliers flagged ({LABELS[sp]:8s}): {flag.sum()}")
    outlier_mask |= flag
df["is_outlier"] = outlier_mask

# 4f. Impute missing month with season midpoint based on count_type
#nests/adults: Nov (11, peak breeding season), chicks → Feb (2)
season_month_map = {"nests": 11, "adults": 11, "chicks": 2}
df["month"] = df.apply(
    lambda r: season_month_map.get(r["count_type"], 11) if pd.isna(r["month"]) else r["month"],
    axis=1
)
print(f"  Remaining missing months  : {df['month'].isna().sum()}")

# 4g. Fill missing vantage with 'unknown'
df["vantage"] = df["vantage"].fillna("unknown")

# 4h. Derived features
df["log_count"]     = np.log1p(df["penguin_count"])
df["decade"]        = (df["year"] // 10 * 10).astype(str) + "s"
df["species_label"] = df["common_name"].map(LABELS)

# 4i. Standardise & normalise count
df["count_zscore"] = StandardScaler().fit_transform(df[["penguin_count"]])
df["count_minmax"] = MinMaxScaler().fit_transform(df[["penguin_count"]])

df.sort_values(["common_name", "year"], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"\n Clean rows: {len(df):,}")


# 5. AFTER SNAPSHOT

print("\nAFTER snapshot")
print(df[["common_name","year","site_name","penguin_count","count_type",
          "log_count","count_zscore","is_outlier"]].head(8).to_string())


# 6. STATISTICAL SUMMARY
print("\nStep 3: Statistical Summary")

summary = df.groupby("common_name")["penguin_count"].agg([
    "count", "mean", "median",
    ("std",      "std"),
    ("var",      "var"),
    ("skew",     "skew"),
    ("kurtosis", lambda x: x.kurtosis()),
    "min", "max",
]).round(2)
print(summary.to_string())
summary.to_csv(f"{OUT}/summary_statistics.csv")

# Annual totals per species
annual = df.groupby(["year", "common_name"])["penguin_count"].sum().reset_index()
annual_wide = annual.pivot(index="year", columns="common_name", values="penguin_count")

print("\nCorrelation between species annual totals:")
corr_mat = annual_wide.corr()
print(corr_mat.round(3))

# Kruskal-Wallis test — are species counts significantly different?
groups = [df.loc[df["common_name"] == sp, "penguin_count"].values for sp in SPECIES]
h, p = kruskal(*groups)
print(f"\nKruskal-Wallis  H={h:.2f}, p={p:.4e}")

# Count type breakdown
print("\nRecords by count_type:")
print(df.groupby(["species_label", "count_type"]).size().unstack(fill_value=0).to_string())


# 7. PCA
pca_input = df[["penguin_count", "log_count", "year", "accuracy", "lat", "lon"]].fillna(0)
pca = PCA(n_components=2)
pcs = pca.fit_transform(StandardScaler().fit_transform(pca_input))
df["PC1"] = pcs[:, 0]
df["PC2"] = pcs[:, 1]
print(f"\nPCA explained variance: PC1={pca.explained_variance_ratio_[0]:.3f}  PC2={pca.explained_variance_ratio_[1]:.3f}")


# 8. VISUALISATIONS

print("\nStep 4: Visualisations")

# Fig 1: Missing values heatmap (raw data, all species)
fig, ax = plt.subplots(figsize=(9, 4))
miss = (raw.isnull().sum() / len(raw) * 100).sort_values()
colors = ["#d73027" if v > 10 else "#fee090" if v > 0 else "#1a9850" for v in miss]
bars = ax.barh(miss.index, miss.values, color=colors)
for bar, val in zip(bars, miss.values):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f"{val:.1f}%", va="center", fontsize=9)
ax.set_xlabel("Missing Values (%)")
ax.set_title("Fig 1 — Missing Values per Column (Raw MAPPPD Data)")
plt.tight_layout()
plt.savefig(f"{OUT}/fig01_missing_values.png", dpi=150); plt.close()
print("  ✓ Fig 1 – Missing values")

#Fig 2: Annual population trends (all 3 species)
fig, ax = plt.subplots(figsize=(13, 5))
for sp in SPECIES:
    sub = annual[annual["common_name"] == sp].sort_values("year")
    ax.plot(sub["year"], sub["penguin_count"], color=PALETTE[sp],
            linewidth=2.2, marker="o", markersize=3.5, label=LABELS[sp])
    z = np.polyfit(sub["year"], sub["penguin_count"], 1)
    ax.plot(sub["year"], np.poly1d(z)(sub["year"]),
            color=PALETTE[sp], linestyle="--", linewidth=1, alpha=0.5)
ax.set_xlabel("Year"); ax.set_ylabel("Total Penguin Count")
ax.set_title("Fig 2 — Annual Population Totals: Gentoo, King, Macaroni (1979–2025)\n"
             "(dashed = linear trend)")
ax.legend(); plt.tight_layout()
plt.savefig(f"{OUT}/fig02_population_trends.png", dpi=150); plt.close()
print("  ✓ Fig 2 – Annual trends")

# ── Fig 3: Boxplot by species
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df, x="species_label", y="penguin_count",
            palette={"Gentoo": "#1b7837", "Macaroni": "#762a83", "King": "#e08214"},
            width=0.5, flierprops={"marker": "x", "markersize": 4}, ax=ax)
ax.set_xlabel("Species"); ax.set_ylabel("Penguin Count (per survey)")
ax.set_title("Fig 3 — Survey Count Distribution by Species\n(with outliers marked as ×)")
plt.tight_layout()
plt.savefig(f"{OUT}/fig03_boxplot_species.png", dpi=150); plt.close()
print("  ✓ Fig 3 – Boxplot")

# ── Fig 4: Log-transformed count distributions
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, sp in zip(axes, SPECIES):
    sub = df[df["common_name"] == sp]["log_count"].dropna()
    sns.histplot(sub, kde=True, bins=25, color=PALETTE[sp], ax=ax)
    ax.set_title(f"{LABELS[sp]}", fontsize=11)
    ax.set_xlabel("log(1 + count)")
fig.suptitle("Fig 4 — Log-Transformed Count Distributions", fontsize=12)
plt.tight_layout()
plt.savefig(f"{OUT}/fig04_log_distributions.png", dpi=150); plt.close()
print("  ✓ Fig 4 – Log distributions")

# ── Fig 5: QQ plots (normality)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, sp in zip(axes, SPECIES):
    sample = df[df["common_name"] == sp]["log_count"].dropna()
    stats.probplot(sample, dist="norm", plot=ax)
    ax.set_title(f"QQ — {LABELS[sp]}", fontsize=10)
fig.suptitle("Fig 5 — QQ Plots: Normality Check on Log Count", fontsize=12)
plt.tight_layout()
plt.savefig(f"{OUT}/fig05_qq_plots.png", dpi=150); plt.close()
print("  ✓ Fig 5 – QQ plots")

# ── Fig 6: Count type breakdown (stacked bar)
ct = df.groupby(["species_label", "count_type"]).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(9, 5))
ct.plot(kind="bar", stacked=True, ax=ax, colormap="Set2", edgecolor="white")
ax.set_xlabel("Species"); ax.set_ylabel("Number of Survey Records")
ax.set_title("Fig 6 — Survey Records by Species and Count Type\n(nests / adults / chicks)")
ax.tick_params(axis="x", rotation=0); ax.legend(title="Count Type")
plt.tight_layout()
plt.savefig(f"{OUT}/fig06_count_type.png", dpi=150); plt.close()
print("  ✓ Fig 6 – Count type breakdown")

# ── Fig 7: Correlation heatmap
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr_mat, annot=True, fmt=".3f", cmap="RdBu_r",
            center=0, square=True, linewidths=1, ax=ax,
            xticklabels=[LABELS[c] for c in corr_mat.columns],
            yticklabels=[LABELS[c] for c in corr_mat.index])
ax.set_title("Fig 7 — Inter-Species Correlation (Annual Totals)")
plt.tight_layout()
plt.savefig(f"{OUT}/fig07_correlation_heatmap.png", dpi=150); plt.close()
print("  ✓ Fig 7 – Correlation heatmap")

# ── Fig 8: 5-year rolling average
fig, ax = plt.subplots(figsize=(13, 5))
for sp in SPECIES:
    sub = annual[annual["common_name"] == sp].sort_values("year").set_index("year")
    roll = sub["penguin_count"].rolling(5, center=True, min_periods=2).mean()
    ax.fill_between(sub.index, sub["penguin_count"], alpha=0.12, color=PALETTE[sp])
    ax.plot(sub.index, roll, color=PALETTE[sp], linewidth=2.5, label=f"{LABELS[sp]} (5yr avg)")
ax.set_xlabel("Year"); ax.set_ylabel("Penguin Count")
ax.set_title("Fig 8 — 5-Year Rolling Average (shaded = raw annual totals)")
ax.legend(); plt.tight_layout()
plt.savefig(f"{OUT}/fig08_rolling_average.png", dpi=150); plt.close()
print("  ✓ Fig 8 – Rolling average")

# ── Fig 9: Spatial distribution of survey sites
fig, ax = plt.subplots(figsize=(11, 6))
for sp in SPECIES:
    sub = df[df["common_name"] == sp].drop_duplicates("site_id")
    ax.scatter(sub["lon"], sub["lat"], color=PALETTE[sp], alpha=0.6,
               s=40, label=LABELS[sp], edgecolors="white", linewidths=0.4)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("Fig 9 — Unique Survey Sites by Species")
ax.legend(); plt.tight_layout()
plt.savefig(f"{OUT}/fig09_survey_locations.png", dpi=150); plt.close()
print("  ✓ Fig 9 – Survey sites map")

# ── Fig 10: PCA scatter
fig, ax = plt.subplots(figsize=(8, 6))
for sp in SPECIES:
    sub = df[df["common_name"] == sp]
    ax.scatter(sub["PC1"], sub["PC2"], color=PALETTE[sp], s=15,
               alpha=0.5, label=LABELS[sp])
ev = pca.explained_variance_ratio_
ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}% variance)")
ax.set_title("Fig 10 — PCA of Survey Features by Species")
ax.legend(); plt.tight_layout()
plt.savefig(f"{OUT}/fig10_pca.png", dpi=150); plt.close()
print("  ✓ Fig 10 – PCA")

# ── Fig 11: Year-over-year % change
fig, ax = plt.subplots(figsize=(13, 5))
for sp in SPECIES:
    sub = annual[annual["common_name"] == sp].sort_values("year").copy()
    sub["pct_change"] = sub["penguin_count"].pct_change() * 100
    ax.plot(sub["year"], sub["pct_change"], color=PALETTE[sp],
            linewidth=1.5, alpha=0.85, label=LABELS[sp])
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Year"); ax.set_ylabel("YoY Change (%)")
ax.set_title("Fig 11 — Year-Over-Year Population Growth Rate (%)")
ax.legend(); plt.tight_layout()
plt.savefig(f"{OUT}/fig11_yoy_change.png", dpi=150); plt.close()
print("  ✓ Fig 11 – YoY change")

# Fig 12: Before vs After data quality
three_raw = raw[raw["common_name"].isin(SPECIES)]
metrics  = ["Total Rows\n(3 species)", "Missing\nCounts", "Zero\nCounts",
            "Duplicates", "Outliers\nFlagged"]
before_v = [
    len(three_raw),
    int(three_raw["penguin_count"].isna().sum()),
    int((three_raw["penguin_count"] == 0).sum()),
    int(three_raw.duplicated(subset=["site_id","common_name","year","count_type"]).sum()),
    0,
]
after_v = [len(df), 0, 0, 0, int(df["is_outlier"].sum())]
x = np.arange(len(metrics)); w = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - w/2, before_v, w, label="Before", color="#d73027", alpha=0.85)
ax.bar(x + w/2, after_v,  w, label="After",  color="#1a9850", alpha=0.85)
for i, (b, a) in enumerate(zip(before_v, after_v)):
    ax.text(i - w/2, b + 1, str(b), ha="center", fontsize=9)
    ax.text(i + w/2, a + 1, str(a), ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylabel("Count")
ax.set_title("Fig 12 — Before vs. After: Data Quality Comparison")
ax.legend(); plt.tight_layout()
plt.savefig(f"{OUT}/fig12_before_after.png", dpi=150); plt.close()
print(" Fig 12 – Before/After")

# 9. SAVE OUTPUTS
df.to_csv(f"{OUT}/mapppd_cleaned.csv", index=False)
three_raw.to_csv(f"{OUT}/mapppd_raw_3species.csv", index=False)
annual.to_csv(f"{OUT}/annual_totals.csv", index=False)
summary.to_csv(f"{OUT}/summary_statistics.csv")

print(f"""

Raw rows (3 species) - Clean : {len(three_raw):,} - {len(df):,}              
Years covered         : 1979 – 2025
Source                : MAPPPD
""")

── Step 1: Data Collection ──
  Using cached file: DownloadAll.csv
  Raw shape  : (5407, 15)
  Columns    : ['site_name', 'site_id', 'cammlr_region', 'longitude_epsg_4326', 'latitude_epsg_4326', 'common_name', 'day', 'month', 'year', 'season_starting', 'penguin_count', 'accuracy', 'count_type', 'vantage', 'reference']
  All species: ['adelie penguin', 'chinstrap penguin', 'macaroni penguin', 'gentoo penguin', 'king penguin', 'emperor penguin']

── BEFORE snapshot (3 species) ──
  Rows          : 1,707
  Missing values:
site_name                0
site_id                  0
cammlr_region            0
longitude_epsg_4326      0
latitude_epsg_4326       0
common_name              0
day                    342
month                  308
year                     0
season_starting          0
penguin_count            5
accuracy                 5
count_type               0
vantage                 15
reference                0
         common_name  year                           site_name  pengui